## DAY 2- RAG 

In [ ]:
from groq import Groq
import datetime
import json

client = Groq(api_key="Your-key")

# YOUR TOOLS — normal Python functions
def get_date():
    return str(datetime.date.today())

def calculator(expression: str):
    return str(eval(expression))

def get_weather(city: str):
    # Fake for now — later you'd call a real weather API
    return f"Weather in {city}: 32°C, Humid"

# Tell LLM these tools exist
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_date",
            "description": "Get today's date",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    }
]

# LLM decides which tool to call
messages = [{"role": "user", "content": "What is today's date? What is 847 * 23? What's the weather in Mumbai?"}]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

# Execute whatever tool LLM chose
tool_map = {"get_date": get_date, "calculator": calculator, "get_weather": get_weather}

for tool_call in response.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    result = tool_map[name](**args) if args else tool_map[name]()
    print(f"Tool called: {name}")
    print(f"Result: {result}\n")

Tool called: get_date
Result: 2026-05-07

Tool called: calculator
Result: 19481

Tool called: get_weather
Result: Weather in Mumbai: 32°C, Humid



In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", 
               "chromadb", "sentence-transformers", "pypdf"], 
               capture_output=False)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", 
               "sentence-transformers"], capture_output=False)

In [4]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

# STEP 1: Read resume
reader = PdfReader("Neha_Resume.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text()
print("Resume loaded:", len(full_text), "characters")

# STEP 2: Split into chunks
chunks = []
for i in range(0, len(full_text), 350):
    chunk = full_text[i:i+350]
    if chunk.strip():
        chunks.append(chunk)
print(f"Split into {len(chunks)} chunks")

# STEP 3: Embeddings
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks).tolist()
print("Embeddings created")

# STEP 4: Store in vector DB
db = chromadb.Client()
collection = db.create_collection("neha_resume")
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print("Stored in vector DB")

# STEP 5: Ask questions
client = Groq(api_key="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")

def ask_resume(question):
    q_embed = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embed, n_results=3)
    context = "\n".join(results['documents'][0])
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Answer only using the context provided. Be concise."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

# Test
print("\n--- RAG on your resume ---\n")
print("Q: What internships has Neha done?")
print(ask_resume("What internships has Neha done?"))
print("\nQ: What tech stack does Neha know?")
print(ask_resume("What tech stack does Neha know?"))
print("\nQ: What projects has Neha built?")
print(ask_resume("What projects has Neha built?"))

Resume loaded: 3799 characters
Split into 11 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\kanwa\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kanwa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created
Stored in vector DB

--- RAG on your resume ---

Q: What internships has Neha done?
Neha has done the following internship: 
1. Software Engineering Intern at GradPipe.

Q: What tech stack does Neha know?
Large Language Models (LLMs), Prompt Engineering, and backend development (implied by GitHub activity analysis).

Q: What projects has Neha built?
Neha built and improved features for an AI-driven talent marketplace, and developed a backend to analyze GitHub activity, project repositories, and developer profiles for hiring insights.


In [ ]:
print("What ML experience does Neha have?")

while True:
    question = input("You: ")
    if question.lower() == 'quit':
        print("Done!")
        break
    answer = ask_resume(question)
    print(f"\nAI: {answer}\n")
    print("─" * 50 + "\n")

What ML experience does Neha have?


You:  Heart disease model



AI: The Heart Disease Modeling project involved: 
1. End-to-end data preprocessing and EDA on the UCI Heart Disease dataset.
2. Training and evaluating Logistic Regression and Random Forest models.
3. Applying PCA and K-Means clustering to identify subgroups based on clinical features and cardiovascular risk factors.

──────────────────────────────────────────────────



You:  LLM



AI: Large Language Models (LLMs) are one of the technologies mentioned, along with Prompt Engineering.

──────────────────────────────────────────────────



...


AI: React.js, JavaScript, HTML, CSS, React Hooks.

──────────────────────────────────────────────────



You:  Full stack



AI: Full-stack YouTube clone web app built using React.js, Django, and REST API.

──────────────────────────────────────────────────



You:  Internship at ISB



AI: At ISB, Hyderabad, Neha was an Executive Profiling and Behavioral Analysis Research Intern from Jan' 25 to May' 25. She worked under Prof. Abhishek Kathuria and:

* Automated metadata extraction from 4,000+ profiles
* Built Python scripts for quantitative text analysis of executive summaries

──────────────────────────────────────────────────

